# Set-up

In [ ]:
# Import packages
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from torch.utils.data import DataLoader, Subset
import os, sys, json, time, glob, warnings
from matplotlib.colors import SymLogNorm
from matplotlib.ticker import FormatStrFormatter

# Add parent directory to sys.path for module imports
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import modules
from src.data.dataset import LidarS2Dataset
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddpm, p_sample_loop_ddim, p_sample_loop_plms
from src.utils.metrics import normalize_batch
import rasterio

# Ignore warnings
warnings.filterwarnings("ignore")

# Load Trained Model

In [ ]:
# Config

REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
CKPT_PATH = os.path.join(REPO_DIR, "models", "cosine_k6_att_best.pth")
TEST_S2_DIR = [
    os.path.join(REPO_DIR, "input_data", "s2_patches_pondinlet"),
    os.path.join(REPO_DIR, "input_data", "s2_patches_tuk"),
    os.path.join(REPO_DIR, "input_data", "s2_patches_cambridge"),
]
TEST_LIDAR_DIR = [
    os.path.join(REPO_DIR, "input_data", "lidar_patches_pondinlet"),
    os.path.join(REPO_DIR, "input_data", "lidar_patches_tuk"),
    os.path.join(REPO_DIR, "input_data", "lidar_patches_cambridge"),
]
OUT_DIR = os.path.join(REPO_DIR, "figures", "test")

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT_PATH, map_location=device)
config = ckpt["config"]  # original training config

# Override paths for test data
config["data"]["s2_dir"]   = TEST_S2_DIR
config["data"]["lidar_dir"] = TEST_LIDAR_DIR
config["logging"]["output_dir"] = OUT_DIR
config["system"]["device"] = device

In [3]:
# Create test dataset 

test_dataset = LidarS2Dataset(
    lidar_dirs=TEST_LIDAR_DIR,
    s2_dirs=TEST_S2_DIR,
    context_k=config["training"]["context_k"],
    randomize_context=False,
    augment=False,
    debug=False,
    split="val"  
)

Prepared 14984 matched LiDAR↔S2 groups (k=6).


In [4]:
# Load in trained model

model = ConditionalUNet(
    in_channels=1,
    cond_channels=4 * config["training"]["context_k"],
    attr_dim=8 * config["training"]["context_k"],
    base_channels=config["model"]["base_channels"],
    embed_dim=config["model"]["embed_dim"],
    unet_depth=config["model"]["unet_depth"],
    attention_variant=config["model"]["attention_variant"],
    cond_k=config["training"]["context_k"]
).to(device)
model.load_state_dict(ckpt["model_state_dict"])

# Load in diffusion scheduler from trained model
if config["training"]["noise_schedule"] == "linear":
    scheduler = LinearDiffusionScheduler(config["training"]["timesteps"], device=device)
else:
    scheduler = CosineDiffusionScheduler(config["training"]["timesteps"], device=device)


# Inference

In [23]:
def run_inference(model, val_dataset, config, scheduler=None, out_path=None):
    """
    Run inference and create a clean reconstruction figure with:
      - 2 Sentinel-2 context rows
      - GT LiDAR
      - Pred LiDAR
      - Error
      - Overlaid GT vs Pred patch PDF
    """
    print("\n" + "=" * 60)
    print("RUNNING RECONSTRUCTION EVALUATION")
    print("=" * 60)

    torch.manual_seed(config["system"].get("seed", 42))
    val_dataset.split = "val"
    model.eval()

    # ------------------------------------------------------------------
    # Hand-picked tiles in desired display order:
    # Pond Inlet A, Pond Inlet B, Tuk A, Tuk B, Cambridge A, Cambridge B
    # ------------------------------------------------------------------
    eval_pids = ["05275", "05301", "12518", "12551", "14200", "14500"]

    eval_indices = [
        i for i, sample in enumerate(val_dataset.samples)
        if sample["tile_id"] in eval_pids
    ]

    if len(eval_indices) != len(eval_pids):
        print(
            f"Warning: Found {len(eval_indices)} out of {len(eval_pids)} requested evaluation patches."
        )

    # Reorder subset to match eval_pids exactly
    tileid_to_index = {
        val_dataset.samples[i]["tile_id"]: i
        for i in eval_indices
    }
    ordered_indices = [tileid_to_index[pid] for pid in eval_pids if pid in tileid_to_index]

    eval_subset = Subset(val_dataset, ordered_indices)
    eval_loader = DataLoader(eval_subset, batch_size=len(eval_subset), shuffle=False)
    batch = next(iter(eval_loader))

    device = config["system"]["device"]
    s2 = batch["s2"].to(device)
    lidar = batch["lidar"].to(device)   # demeaned residual LiDAR
    attrs = batch["attrs"].to(device)
    chosen_ids_batch = batch["chosen_ids"]
    tile_ids_batch = batch["tile_id"]

    B = lidar.size(0)

    all_samplers = {
        "ddpm": lambda m, s, c, a, d: p_sample_loop_ddpm(m, scheduler, s, c, a, d) if scheduler else None,
        "ddim": lambda m, s, c, a, d: p_sample_loop_ddim(m, scheduler, s, c, a, d) if scheduler else None,
        "plms": lambda m, s, c, a, d: p_sample_loop_plms(m, scheduler, s, c, a, d) if scheduler else None,
    }
    requested_methods = config["evaluation"]["sampling_methods"]
    p_samplers = {m: all_samplers[m] for m in requested_methods if m in all_samplers}

    if not p_samplers:
        print("No valid samplers available")
        return

    # Column headers
    column_titles = [
        "Pond Inlet\nSample A",
        "Pond Inlet\nSample B",
        "Tuktoyaktuk\nSample A",
        "Tuktoyaktuk\nSample B",
        "Cambridge\nSample A",
        "Cambridge\nSample B",
    ]

    # Lookup for S2 visualization
    tileid_to_s2dir = {s["tile_id"]: s["s2_group_dir"] for s in val_dataset.samples}

    for sampler_name, sampler_func in p_samplers.items():
        print(f"\nSampling method: {sampler_name}")

        with torch.no_grad():
            if torch.cuda.is_available():
                torch.cuda.synchronize()

            pred = sampler_func(model, lidar.shape, s2, attrs, device)

            if torch.cuda.is_available():
                torch.cuda.synchronize()

        # Demeaned residual versions for plotting
        gt = lidar.cpu()
        pred = pred.cpu()

        # ------------------------------------------------------------------
        # Figure layout
        # ------------------------------------------------------------------
        num_s2_patches = 2
        num_data_rows = 4  # GT, Pred, Error, PDF
        total_rows = num_s2_patches + num_data_rows
        num_cols = B

        tile_size_inches = 4.8
        fig_w = num_cols * tile_size_inches + 2.2
        fig_h = total_rows * tile_size_inches + 1.2

        fig, axes = plt.subplots(
            total_rows,
            num_cols,
            figsize=(fig_w, fig_h),
            squeeze=False,
            gridspec_kw={
                "hspace": 0.03,
                "wspace": 0.03,
                "height_ratios": [1, 1, 1, 1, 1, 0.99]  
            },
        )

        # Shared ranges for image plots
        max_abs_resid = torch.quantile(
            torch.abs(torch.cat([gt.flatten(), pred.flatten()])),
            0.995
        ).item()

        signed_error = pred.squeeze(1) - gt.squeeze(1)
        max_abs_error = torch.quantile(
            torch.abs(signed_error.flatten()),
            0.995
        ).item()

        norm = SymLogNorm(
            linthresh=0.1,
            linscale=1.0,
            vmin=-max_abs_resid,
            vmax=max_abs_resid,
            base=10,
        )

        # ------------------------------------------------------------------
        # Prepare S2 visualization
        # ------------------------------------------------------------------
        s2_viz_data = []
        for i in range(B):
            tile_id = tile_ids_batch[i]
            s2_group_dir = tileid_to_s2dir.get(tile_id)

            if s2_group_dir is None:
                print(f"Warning: no s2_group_dir found for tile {tile_id}; skipping S2 viz.")
                s2_viz_data.append(
                    [np.zeros((*gt.shape[-2:], 3), dtype=np.float32) for _ in range(num_s2_patches)]
                )
                continue

            chosen_ids = chosen_ids_batch[i].tolist()
            processed_sample = []

            for t_id in chosen_ids[:num_s2_patches]:
                s2_path = os.path.join(s2_group_dir, f"t{t_id}.tif")
                with rasterio.open(s2_path) as src:
                    arr = torch.from_numpy(src.read()[:4].astype(np.float32))

                rgb = arr[[0, 1, 2], :, :]
                rgb = normalize_batch(rgb.unsqueeze(0)).squeeze(0)
                rgb = F.interpolate(
                    rgb.unsqueeze(0),
                    size=gt.shape[-2:],
                    mode="bilinear",
                    align_corners=False
                ).squeeze(0)
                processed_sample.append(rgb.permute(1, 2, 0).numpy())

            while len(processed_sample) < num_s2_patches:
                processed_sample.append(np.zeros((*gt.shape[-2:], 3), dtype=np.float32))

            s2_viz_data.append(processed_sample)

        # Left-side row labels
        row_titles = [
            "S2 Sample #1",
            "S2 Sample #2",
            "GT LiDAR",
            "Pred LiDAR",
            "Error",
            "Patch PDF",
        ]

        # ------------------------------------------------------------------
        # Plot columns
        # ------------------------------------------------------------------
        for col in range(num_cols):
            gt_i = gt[col].squeeze().numpy()
            pred_i = pred[col].squeeze().numpy()
            err_i = (pred[col].squeeze() - gt[col].squeeze()).numpy()

            # Column headers
            axes[0, col].set_title(
                column_titles[col],
                fontsize=22,
                fontweight="bold",
                pad=18
            )

            # S2 rows
            for row in range(num_s2_patches):
                ax = axes[row, col]
                ax.imshow(s2_viz_data[col][row])
                ax.axis("off")

            # GT
            row_gt = num_s2_patches
            ax = axes[row_gt, col]
            im_gt = ax.imshow(gt_i, cmap="RdBu_r", norm=norm)
            ax.axis("off")

            # Pred
            row_pred = num_s2_patches + 1
            ax = axes[row_pred, col]
            im_pred = ax.imshow(pred_i, cmap="RdBu_r", norm=norm)
            ax.axis("off")

            # Error
            row_err = num_s2_patches + 2
            ax = axes[row_err, col]
            im_err = ax.imshow(err_i, cmap="seismic", vmin=-max_abs_error, vmax=max_abs_error)
            ax.axis("off")

            # PDF row: overlaid GT vs Prediction
            row_pdf = num_s2_patches + 3
            ax = axes[row_pdf, col]

            gt_vals = gt_i[np.isfinite(gt_i)].ravel()
            pred_vals = pred_i[np.isfinite(pred_i)].ravel()

            if len(gt_vals) > 1 and len(pred_vals) > 1:
                all_vals = np.concatenate([gt_vals, pred_vals])

                vmin_pdf, vmax_pdf = np.quantile(all_vals, [0.01, 0.99])
                if np.isclose(vmin_pdf, vmax_pdf):
                    vmin_pdf = all_vals.min()
                    vmax_pdf = all_vals.max()

                if np.isclose(vmin_pdf, vmax_pdf):
                    vmin_pdf -= 1e-3
                    vmax_pdf += 1e-3

                bins = np.linspace(vmin_pdf, vmax_pdf, 60)

                gt_pdf, edges = np.histogram(gt_vals, bins=bins, density=True)
                pred_pdf, _ = np.histogram(pred_vals, bins=bins, density=True)
                centers = 0.5 * (edges[:-1] + edges[1:])

                ax.plot(centers, gt_pdf, color="darkblue", lw=2, label="GT")
                ax.plot(centers, pred_pdf, color="red", lw=2, alpha=0.95, label="Pred")
                ax.fill_between(centers, gt_pdf, color="darkblue", alpha=0.15)
                ax.fill_between(centers, pred_pdf, color="red", alpha=0.15)

                ax.set_xlim(vmin_pdf, vmax_pdf)
                ymax = max(np.max(gt_pdf), np.max(pred_pdf))
                ax.set_ylim(0, ymax * 1.08 if ymax > 0 else 1)

            ax.grid(True, alpha=0.25)
            ax.tick_params(labelsize=12)
            ax.set_xlabel("Residual (m)", fontsize=20)

            if col == 0:
                ax.set_ylabel("Density", fontsize=20)
                ax.legend(fontsize=20, frameon=False, loc="upper right")
            else:
                ax.set_yticklabels([])

            # Force square PDF panel
            ax.set_box_aspect(1)
            ax.set_anchor("C")

        # Left row labels
        for row in range(total_rows):
            ax = axes[row, 0]
            ax.text(
                -0.18,
                0.5,
                row_titles[row],
                ha="right",
                va="center",
                transform=ax.transAxes,
                fontsize=22,
                fontweight="bold",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, boxstyle="round,pad=0.25"),
            )

        # ------------------------------------------------------------------
        # Colorbars
        # ------------------------------------------------------------------
        ax_gt_anchor = axes[row_gt, num_cols - 1]
        ax_err_anchor = axes[row_err, num_cols - 1]

        axins_gt = inset_axes(
            ax_gt_anchor,
            width="5%",
            height="205%",
            loc="right",
            bbox_to_anchor=(0.1, -0.52, 1, 1),
            bbox_transform=ax_gt_anchor.transAxes,
        )
        cbar_gt = fig.colorbar(im_gt, cax=axins_gt)
        cbar_gt.set_ticks([
            -max_abs_resid,
            -0.1 * max_abs_resid,
            0.0,
            0.1 * max_abs_resid,
            max_abs_resid,
        ])
        cbar_gt.formatter = FormatStrFormatter("%.2f")
        cbar_gt.update_ticks()
        cbar_gt.ax.tick_params(labelsize=12)

        axins_err = inset_axes(
            ax_err_anchor,
            width="5%",
            height="100%",
            loc="right",
            bbox_to_anchor=(0.1, 0, 1, 1),
            bbox_transform=ax_err_anchor.transAxes,
        )
        cbar_err = fig.colorbar(im_err, cax=axins_err)
        cbar_err.formatter = FormatStrFormatter("%.2f")
        cbar_err.ax.tick_params(labelsize=12)

        plt.subplots_adjust(left=0.09, bottom=0.06, top=0.90, right=0.95)

        if out_path is not None:
            # If multiple samplers are used, append sampler name
            save_path = out_path
            if len(p_samplers) > 1:
                root, ext = os.path.splitext(out_path)
                save_path = f"{root}_{sampler_name}{ext}"

            fig.savefig(save_path, dpi=300, bbox_inches="tight")
            print(f"Saved visualization to {save_path}")

        plt.show()

        plt.close(fig)

In [ ]:
run_inference(model, test_dataset, config, scheduler=scheduler, out_path=os.path.join(REPO_DIR, "figures", "patch_recontructions_pdfs.png"))